In [11]:
import csv
from Bio import SeqIO
from Bio.SeqFeature import SeqFeature, FeatureLocation
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import re

Вывод id-шников, у которых везде NA-NA

In [12]:
import pandas as pd
df = pd.read_csv("AV_2_orf.txt")
result = df[(df["1A"] == "NA-NA") & (df["1B"] == "NA-NA") & (df["2"] == "NA-NA")]
id_list = result["id"].tolist()
print(id_list)
pd.DataFrame(id_list, columns=["id"]).to_csv("all_NA_ids.csv", index=False)

['PX095754', 'PX095696', 'PX113257', 'PX113255', 'PX113254', 'PV793827', 'PV793829', 'PV793828', 'PV793826', 'PV793833', 'PV793831', 'PQ421854', 'PQ421853', 'PQ421852', 'PQ421851', 'PQ421848', 'PQ421847', 'PQ421846', 'PQ421845', 'PQ421844', 'PQ421843', 'PQ421842', 'PQ421841', 'PP211271', 'PP211223', 'PP211222', 'PP211221', 'PG036827', 'PG036826', 'PG036825', 'PG036824', 'PG036823', 'PG036822', 'PG036821', 'PG036820', 'PQ150499', 'PD223317', 'PD223316', 'PD222915', 'PD222914', 'PL092599', 'PL092591', 'PL092587', 'PL092412', 'PL092404', 'PL092400', 'PL092225', 'PL092217', 'PL092213', 'PL190248', 'PQ573797', 'PQ573796', 'PQ161557', 'PQ055527', 'PQ335170', 'PP272729', 'PQ110289', 'PP512783', 'OR951098', 'OR951097', 'OR951096', 'OR951094', 'OR951092', 'OR951091', 'OR951090', 'OR951089', 'OR951088', 'OR951087', 'OR951086', 'OR951083', 'OR951082', 'OR951078', 'OR951077', 'OR951076', 'OR951075', 'OR951074', 'OR951073', 'OR951072', 'OR951070', 'OR951069', 'OR951068', 'OR951067', 'OR951065', 'OR

Скрипт поиска слов из 1ab_2_final_note.csv в definition файла AV_2.gb, результат записываем в extra_genes.csv

In [13]:
gb_file = "AV_2.gb"
csv_file = "1ab_2_final_note.csv"
output_file = "extra_genes.csv"
id_list_set = set(id_list)

csv_strings = {}
with open(csv_file, encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if row:
            csv_strings[row[0].strip()] = row[1]
print(csv_strings)
in_def = []
for record in SeqIO.parse(gb_file, "genbank"):
    record_id_root = record.id.split(".")[0]
    if record_id_root not in id_list_set:
        continue
    definition = record.description
    if definition:
        def_lower = definition.lower()
        for csv_str in csv_strings:
            pattern = r"(?<!\w)" + re.escape(csv_str.lower()) + r"(?!\w)"
            if re.search(pattern, def_lower):
                in_def.append((record.id, csv_str, csv_strings[csv_str]))
                break
extra_genes = {}
for rec_id, match, gene in in_def:
    extra_genes[rec_id] = gene

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "gene"])
    for rec_id, gene in extra_genes.items():
        writer.writerow([rec_id, gene])

{'0RF1b': '1B', '1a': '1A', '1ab': '1AB', '1b': '1B', '3C-like serine protease motif': '1A', '3C-like serine proteinase': '1A', "5' translation start site undetermined; contains RNA-dependent RNA polymerase motif": '1B', 'CAstV ORF1b': '1B', 'CP': '2', 'Capsid': '2', 'Capsid precursor protein': '2', 'HAstV1': '', 'HAstV1; ORF1b/ORF2; ORF1b': '1B', 'HAstV1; ORF2': '2', 'MLB2; ORF1b/ORF2; ORF1b': '1B', 'MLB2; ORF2': '2', 'NS': '1A', 'NS1': '1AB', 'NSP1a': '1A', 'NSP1a protein': '1A', 'NSP1ab': '1AB', 'NSP1ab protein': '1AB', 'NSP1b; ORF1B': '1B', 'Non structural gene; 2b like nuclear targetting sequence, serine protease': '1A', 'Non-structural gene composed of two sections 1a and 1b separated by ribosomal frame shift motif at appx position 2840 leading to expression of RNA-dependent RNA polymerase.': '1AB', 'Non-structural polyprotein 1A': '1A', 'Non-structural polyprotein 1AB': '1AB', 'Non-structural protein': '1AB', 'Nsp1a': '1A', 'OEF1b': '1B', 'OFR1ab': '1AB', 'OFR2': '2', 'ORF 1a': 

Теперь проходимся по файлу AV_2.gb и в найденных idшниках добавляем product = gene

In [14]:

gb_file = "AV_2.gb"
extra_genes_file = "extra_genes.csv"
output_file = "AV_2_modified.gb"

extra_genes = {}
with open(extra_genes_file, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rec_id = row["id"].split('.')[0].strip()  # ID до точки
        gene = row["gene"].strip()
        extra_genes[rec_id] = gene

records = list(SeqIO.parse(gb_file, "genbank"))
for rec in records:
    rec_id_root = rec.id.split('.')[0]
    if rec_id_root in extra_genes:
        gene_name = extra_genes[rec_id_root]
        feature = SeqFeature(
            FeatureLocation(0, len(rec.seq)),
            type="CDS",
            qualifiers={"product": gene_name}
        )
        rec.features.append(feature)

SeqIO.write(records, output_file, "genbank")


/opt/anaconda3/lib/python3.13/site-packages/Bio/SeqIO/InsdcIO.py:600: BiopythonWarning: Annotation 'BioSample: SAMN34109731, SAMN34109732, SAMN34109733, SAMN34109815, SAMN34109863' too long
  warnings.warn(f"Annotation {text!r} too long", BiopythonWarning)
/opt/anaconda3/lib/python3.13/site-packages/Bio/SeqIO/InsdcIO.py:600: BiopythonWarning: Annotation 'Sequence Read Archive: SRR24113121, SRR24113120, SRR24113119, SRR24113058, SRR24113208' too long
  warnings.warn(f"Annotation {text!r} too long", BiopythonWarning)
/opt/anaconda3/lib/python3.13/site-packages/Bio/SeqIO/InsdcIO.py:600: BiopythonWarning: Annotation 'BioSample: SAMN34109700, SAMN34109701, SAMN34109702, SAMN34109703, SAMN34109704' too long
  warnings.warn(f"Annotation {text!r} too long", BiopythonWarning)
/opt/anaconda3/lib/python3.13/site-packages/Bio/SeqIO/InsdcIO.py:600: BiopythonWarning: Annotation 'Sequence Read Archive: SRR24113301, SRR24113300, SRR24113299, SRR24113298, SRR24113297' too long
  warnings.warn(f"Annotat

15566

Вытаскиваю айдишники с пустыми координатами, несмотря на все манипуляции выше

In [5]:
import pandas as pd
df = pd.read_csv("AV_2_modified_orf.txt")
result = df[(df["1A"] == "NA-NA") &Y08632 (df["1B"] == "NA-NA") & (df["2"] == "NA-NA")]
id_list_modified = result["id"].tolist()
print(len(id_list_modified))

NameError: name 'Y08632' is not defined